In [16]:
# Setup notebook
from pathlib import Path
import pandas as pd
import numpy as np
from statsmodels.tsa.deterministic import CalendarFourier, DeterministicProcess
from sklearn.linear_model import Ridge

# 1. SETUP & PATHS

In [17]:
# Load Train data
comp_dir = Path('../input/competitions/store-sales-time-series-forecasting')

train = pd.read_csv(
    comp_dir / 'train.csv',
    usecols=['store_nbr', 'family', 'date', 'sales', 'onpromotion'],
    dtype={
        'store_nbr': 'category',
        'family': 'category',
        'sales': 'float32',
        'onpromotion': 'uint32',
    },
    parse_dates=['date'],
)
train['date'] = train.date.dt.to_period('D')
train = train.set_index(['store_nbr', 'family', 'date']).sort_index()
train.head()

sales  onpromotion
store_nbr family     date                          
1         AUTOMOTIVE 2013-01-01    0.0            0
                     2013-01-02    2.0            0
                     2013-01-03    3.0            0
                     2013-01-04    3.0            0
                     2013-01-05    5.0            0

In [18]:
# Load test data
test = pd.read_csv(
    comp_dir / 'test.csv',
    dtype={
        'store_nbr': 'category',
        'family': 'category',
        'onpromotion': 'uint32',
    },
    parse_dates=['date']
)
test['date'] = test.date.dt.to_period('D')
test = test.set_index(['store_nbr', 'family', 'date']).sort_index()

test.head()

id  onpromotion
store_nbr family     date                            
1         AUTOMOTIVE 2017-08-16  3000888            0
                     2017-08-17  3002670            0
                     2017-08-18  3004452            0
                     2017-08-19  3006234            0
                     2017-08-20  3008016            0

In [19]:
family_sales = (
    train['sales']
    .groupby(['family', 'date'], observed=False)
    .mean()
    .unstack('family')
    .loc['2017']
)
if isinstance(family_sales.columns, pd.MultiIndex):
    family_sales.columns = family_sales.columns.get_level_values(-1)

In [20]:
# # Unstack training sales to Wide Format (Rows: Dates, Columns: Product Families) for 2017 data
# family_sales = (
#     train
#     .groupby(['family', 'date'], observed=False)
#     .mean()
#     .unstack('family')
#     .loc['2017']
# )
# family_sales.head()


In [21]:
# Unstack onpromotion for training set and ensure single-level column names
promo_train = (
    train['onpromotion']
    .groupby(['family', 'date'], observed=False)
    .mean()
    .unstack('family')
    .loc['2017']
)
# Drop top column level if MultiIndex exists so columns are just family names
if isinstance(promo_train.columns, pd.MultiIndex):
    promo_train.columns = promo_train.columns.get_level_values(-1)

# Unstack onpromotion for test set and ensure single-level column names
promo_test = (
    test['onpromotion']
    .groupby(['family', 'date'], observed=False)
    .mean()
    .unstack('family')
)
if isinstance(promo_test.columns, pd.MultiIndex):
    promo_test.columns = promo_test.columns.get_level_values(-1)

In [22]:
# # Unstack onpromotion for training set to match the Wide Format family sales columns for 2017
# promo_train = train['onpromotion'].groupby(['family', 'date'], observed=False).mean().unstack('family').loc['2017']

# # Unstack onpromotion for test set across all families for the 16-day evaluation window
# promo_test = test['onpromotion'].groupby(['family', 'date'], observed=False).mean().unstack('family')

# 2. FEATURE ENGINEERING FUNCTIONS

## A. Oil Data (Interpolation + Lags)

In [23]:
# Load crude oil dataset and set date as the DataFrame index
oil = pd.read_csv(comp_dir / 'oil.csv', parse_dates=['date']).set_index('date')
# Generate a complete daily date range from start of oil data through test set end (Aug 31, 2017)
full_range = pd.date_range(start=oil.index.min(), end='2017-08-31', freq='D')
# Reindex oil DataFrame across the full date range to expose weekend and holiday missing dates
oil = oil.reindex(full_range)
# Fill missing oil prices using linear interpolation across missing days and backfill initial NaNs
oil['dcoilwtico'] = oil['dcoilwtico'].interpolate(method='linear').bfill()
# Convert oil datetime index to daily Period format ('D') to match training matrix
oil.index = oil.index.to_period('D')

# Shift oil price by 16 days so every test day uses a historical, known oil price
oil['oil_lag_16'] = oil['dcoilwtico'].shift(16)
# Calculate a 14-day rolling moving average of crude oil prices to capture macro market trends
oil['oil_ma_14'] = oil['dcoilwtico'].rolling(14).mean()
# Backfill any temporary NaN values generated at the beginning of shifting and rolling operations
oil = oil.bfill()

oil.head()

,dcoilwtico,oil_lag_16,oil_ma_14
2013-01-01,93.140000,93.14,93.409286
2013-01-02,93.140000,93.14,93.409286
2013-01-03,92.970000,93.14,93.409286
2013-01-04,93.120000,93.14,93.409286
2013-01-05,93.146667,93.14,93.409286


## B. National Holidays

In [24]:
# Read Ecuadorian holidays dataset and parse date column
holidays = pd.read_csv(
    comp_dir / 'holidays_events.csv', 
    parse_dates=['date']
)

# Convert holiday dates into daily Period format
holidays['date'] = holidays.date.dt.to_period('D')

# Filter specifically for National non-transferred holidays to capture widespread retail impacts
nat_holidays = holidays[(holidays['locale'] == 'National') & (holidays['transferred'] == False)]

# Create a binary pandas Series indicating national holiday occurrences mapped by unique dates
holiday_series = pd.Series(1, index=nat_holidays['date'].unique(), name='is_holiday')

holiday_series

2012-08-10    1
2012-10-12    1
2012-11-02    1
2012-11-03    1
2012-12-21    1
             ..
2017-12-22    1
2017-12-23    1
2017-12-24    1
2017-12-25    1
2017-12-26    1
Freq: D, Name: is_holiday, Length: 160, dtype: int64

## C. Payday & Salary-Week Features

In [25]:
def get_payday_features(period_index):
    # Convert PeriodIndex back to Datetime timestamps to access date attributes (.day, .dayofweek)
    dates = period_index.to_timestamp()
    # Initialize an empty DataFrame indexed by the input time period
    df = pd.DataFrame(index=period_index)
    
    # Check if current day is the standard 15th mid-month payday
    is_mid = (dates.day == 15)
    # Check if current day is the final day of the calendar month
    is_end = dates.is_month_end
    # Check if payday moves to Friday when the 15th falls on a weekend
    is_mid_fri = (dates.day.isin([13, 14])) & (dates.dayofweek == 4)
    # Check if payday moves to Friday when month-end falls on a weekend
    is_end_fri = (dates.day.isin([28, 29, 30])) & (dates.dayofweek == 4) & (~dates.is_month_end)
    
    # Combine all payday conditions into a single binary feature (1 if payday, 0 otherwise)
    df['is_payday'] = (is_mid | is_end | is_mid_fri | is_end_fri).astype(int)
    
    # Binary flag covering salary spending windows (days 14-18, end of month, and start of month)
    df['is_salary_week'] = (
        ((dates.day >= 14) & (dates.day <= 18)) | #(dates.day.between(14, 18)) | --error
        (dates.day >= dates.days_in_month - 1) | 
        (dates.day <= 3)
    ).astype(int)
    
    return df

# 3. BUILD MATRIX X FOR TRAIN & TEST

In [26]:
# Construct annual Fourier seasonality term with 10 harmonic pairs
fourier = CalendarFourier(freq="YE", order=10)

# Initialize deterministic process with linear trend, day-of-week seasonality, and Fourier terms
dp = DeterministicProcess(
    index=family_sales.index,
    constant=True,
    order=1,
    seasonal=True,
    additional_terms=[fourier],
    drop=True
)

# Generate in-sample deterministic feature matrix for training dates
X_train = dp.in_sample()
# Generate out-of-sample deterministic feature matrix for the 16-day test horizon
X_test = dp.out_of_sample(steps=16)


In [27]:
# Helper function to assemble base deterministic features with external economic and calendar indicators
def assemble_features(X_base):
    # Extract current period index from base matrix
    idx = X_base.index
    
    # Extract corresponding oil features using specified 'dcoilwtico' column name
    oil_feats = oil.loc[idx, ['dcoilwtico', 'oil_lag_16', 'oil_ma_14']]
    
    # Extract holiday binary series reindexed to fit current time range, filling non-holidays with 0
    hol_feats = holiday_series.reindex(idx, fill_value=0)
    
    # Extract payday features for current time range
    pay_feats = get_payday_features(idx)
    
    # Join deterministic features, oil prices, holiday indicators, and payday features horizontally
    return X_base.join([oil_feats, hol_feats, pay_feats], how='left')

In [28]:
# Build complete feature matrix for training period
X_train_full = assemble_features(X_train)
# Build complete feature matrix for 16-day testing forecast horizon
X_test_full = assemble_features(X_test)

# 4. TARGET LOG TRANSFORMATION & FIT

In [36]:
# Log-transform target sales values using log1p to align model loss with competition RMSLE metric
y_train_log = np.log1p(family_sales)

# Initialize dictionary to collect predicted sales arrays for each product family
predictions = {}

# Iterate over every product family column to pair family-specific promotion features
for family in family_sales.columns:
    # Copy full training feature matrix to avoid mutating shared base matrix
    X_train_fam = X_train_full.copy()
    # Attach family-specific promotion training values as a dedicated feature column
    X_train_fam['onpromotion'] = promo_train[family]
    
    # Copy full test feature matrix for evaluation
    X_test_fam = X_test_full.copy()
    # Attach family-specific promotion test values as a dedicated feature column
    X_test_fam['onpromotion'] = promo_test[family]
    
    # Instantiate Ridge regression model with L2 regularization strength alpha=1.0
    model = Ridge(alpha=1.0)
    # Fit Ridge model on family-specific feature matrix against log-transformed family sales
    model.fit(X_train_fam, y_train_log[family])
    
    # Generate log-scale sales predictions for current product family over 16-day test period
    pred_log = model.predict(X_test_fam)
    # Convert predictions back to original sales scale using expm1 inverse transformation
    predictions[family] = np.expm1(pred_log)

# Convert prediction dictionary into a structured Wide Format DataFrame (Rows: Dates, Columns: Families)
pred_df = pd.DataFrame(predictions, index=X_test_full.index)
pred_df.index.name = 'date'
pred_df.columns.name = 'family'

In [37]:
pred_df

family,AUTOMOTIVE,BABY CARE,BEAUTY,BEVERAGES,BOOKS,BREAD/BAKERY,CELEBRATION,CLEANING,DAIRY,DELI,...,MAGAZINES,MEATS,PERSONAL CARE,PET SUPPLIES,PLAYERS AND ELECTRONICS,POULTRY,PREPARED FOODS,PRODUCE,SCHOOL AND OFFICE SUPPLIES,SEAFOOD
date,,,,,,,,,,,,,,,,,,,,,
2017-08-16,5.502650,0.079031,8.009607,3905.002275,0.003426,530.695792,11.317129,1121.018235,917.964074,254.236171,...,8.068878,295.571813,710.296069,7.044618,9.899093,305.237886,77.636816,3742.362270,23153.453283,16.645224
2017-08-17,5.555258,0.071409,4.766680,2810.037708,-0.018418,413.880430,13.762575,810.664876,595.473626,226.825000,...,9.563919,336.609286,239.680246,7.151665,9.349802,276.357463,79.928896,1753.082410,26.335769,14.341244
2017-08-18,6.043864,0.109998,5.370481,3329.519838,0.008688,491.988748,17.189226,1018.442277,751.148448,333.654736,...,11.694605,528.404444,315.217443,8.228005,10.807127,534.764184,96.262784,2224.308081,18.705507,25.174210
2017-08-19,8.782621,0.164490,8.125416,4657.147828,0.060519,621.743510,22.935167,1349.564717,1010.602453,369.163679,...,17.956923,411.689876,505.297305,12.061325,15.915206,469.248929,123.926925,2896.247072,24.698946,22.743764
2017-08-20,8.062005,0.152517,7.726252,4666.979333,0.053682,679.426193,14.488669,1226.291482,994.404299,355.455415,...,15.893469,390.454379,527.755856,11.359172,15.325151,447.367008,114.954763,2858.646975,24.307335,21.630842
2017-08-21,6.055440,0.121774,5.756327,3867.139293,0.044558,576.201470,12.761275,1177.785472,899.910436,316.350593,...,13.554297,360.821980,438.375211,8.494988,11.623071,406.396252,107.357563,2704.534629,15.232260,18.346913
2017-08-22,6.068567,0.107444,5.278540,3602.295460,0.064196,552.754697,14.956094,1138.423427,876.336088,308.658133,...,13.166938,374.822609,426.364540,8.346399,11.601285,403.388756,110.662810,3149.265728,12.233457,17.921701
2017-08-23,6.358197,0.130467,5.426225,3894.783386,0.080742,601.396004,17.126813,1235.139371,1072.341207,320.956880,...,13.661543,397.813633,514.969666,9.315863,12.558612,445.045376,121.311076,4376.621928,12.576341,20.416703
2017-08-24,6.364146,0.089343,4.042009,4664.473307,0.055904,481.427146,20.325296,1168.239091,819.205146,299.243496,...,15.760908,454.282223,402.995093,8.727286,11.489951,398.387661,123.900387,2750.730071,7.489451,18.175714


# 5. STACK & SUBMISSION FILE CREATION

In [39]:
# Stack wide prediction matrix into a long series and reset index to columns
y_pred_stacked = pred_df.stack('family').reset_index()

# Rename default column 0 to the target column name 'sales'
y_pred_stacked.rename(columns={0: 'sales'}, inplace=True)

# Merge predicted sales with test set on date and family, selecting only id and sales
submission = test.reset_index().merge(y_pred_stacked, on=['date', 'family'], how='left')[['id', 'sales']]

# Clip negative values to zero to avoid invalid negative sales predictions
submission['sales'] = submission['sales'].clip(lower=0)

# Save the final submission DataFrame to CSV without writing the index
submission.to_csv('submission.csv', index=False)

# Output completion message once submission file is written
print("Pipeline complete! Clean submission file saved to submission.csv")

Pipeline complete! Clean submission file saved to submission.csv
